split into CVs

add correlation-based selection

1) load and analyze variance of descriptors in all the datasets
2) remove descriptors with low variance
3) run typical pipeline with stability and correlation selection but question chat is it ok that every CV fold has different features?
4) add gaussian processes regression and bayesian linear regression 
5) add hyperparameter optimization

when done with all the datasets, analyze which descriptors are most important for each property
select ~20

debug round 5

- optimize parallel script in terms of logging and run on structures

In [92]:
from pathlib import Path
import pandas as pd
import numpy as np
from pathlib import Path
import scipy.stats as stats
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
import glob
import os
pd.set_option('display.max_rows', None)
warnings.filterwarnings('ignore')

from utils.load_results_to_dataframe import load_json_results

# Utils

In [93]:
def load_and_merge(exp_csv_path: Path, results_dir_path: Path, base: str):
    """Load experimental CSV + descriptor JSONs, then inner-join on `name`."""
    df_results = load_json_results(results_dir_path)
    df_results["base"] = base

    exp_df = pd.read_csv(exp_csv_path)

    if "name" not in exp_df.columns:
        raise ValueError(f"Experimental CSV {exp_csv_path} must contain a 'name' column.")
    if "name" not in df_results.columns:
        raise ValueError(
            f"Results loaded from {results_dir_path} are missing a 'name' column for merging."
        )

    try:
        exp_df["name"] = exp_df["name"].astype(int)
    except:
        pass
    try:
        df_results["name"] = df_results["name"].astype(int)
    except:
        pass
    
    exp_df["name"] = exp_df["name"].astype(str)
    df_results["name"] = df_results["name"].astype(str)


    merged_df = exp_df.merge(df_results, on="name", how="inner")
    print(
        f"Merged experimental {Path(exp_csv_path).name} with results '{base}': {len(merged_df)} rows"
    )
    return merged_df


In [94]:
def remove_low_variance_features(
    X: pd.DataFrame,
    feature_cols: list,
    relative_std_threshold: float = 0.01,
    epsilon: float = 1e-8,
):
    """Low-variance feature removal using relative std.

    relative_std = std(feature) / (|mean(feature)| + epsilon)

    Remove features when: relative_std < relative_std_threshold
    (default 0.01, i.e. 1% safe cutoff).

    Parameters
    ----------
    X:
        DataFrame containing the candidate feature columns (typically the training fold only).
    feature_cols:
        List of candidate feature column names.
    relative_std_threshold:
        Cutoff for dropping low-variance features.
    epsilon:
        Small constant to avoid division by zero when mean ~ 0.

    Returns
    -------
    kept_features : list[str]
    removed_features : list[str]
    relative_std : pd.Series
        relative_std indexed by feature name.
    """
    if feature_cols is None:
        feature_cols = []
    feature_cols = list(feature_cols)

    if len(feature_cols) == 0:
        empty = pd.Series(dtype=float)
        return [], [], empty

    X_feat = X[feature_cols].copy()

    # Coerce to numeric so mean/std behave as expected.
    for c in feature_cols:
        X_feat[c] = pd.to_numeric(X_feat[c], errors="coerce")

    mean = X_feat.mean(axis=0, skipna=True)
    std = X_feat.std(axis=0, ddof=0, skipna=True)
    relative_std = std / (mean.abs() + epsilon)

    keep_mask = np.isfinite(relative_std) & (relative_std >= relative_std_threshold)
    kept_features = relative_std.index[keep_mask].tolist()
    removed_features = relative_std.index[~keep_mask].tolist()

    return kept_features, removed_features, relative_std

In [95]:
def _bh_adjust(pvals):
    """Benjamini-Hochberg FDR adjustment. pvals: array-like. NaNs are preserved (not used in adjustment)."""
    p = np.asarray(pvals, dtype=float)
    out = np.full_like(p, np.nan)
    valid = np.isfinite(p)
    if not np.any(valid):
        return p
    p_valid = p[valid]
    n = len(p_valid)
    order = np.argsort(p_valid)
    p_sorted = p_valid[order]
    ratios = n * p_sorted / np.arange(1, n + 1)
    adj_sorted = np.minimum(1, np.minimum.accumulate(ratios[::-1])[::-1])
    rank_of_original = np.argsort(order)
    out[valid] = adj_sorted[rank_of_original]
    return out

def calculate_correlations_and_plot(
    merged_df,
    target_col,
    p_threshold=0.05,
    fdr_alpha=0.05,
    use_fdr=True,
    normalize=False,
    make_plots=False,
    correlation_threshold=None, 
):
    """Compute Spearman correlations for one or multiple targets.

    Parameters
    ----------
    target_col : str or list[str]
        Target column name(s).

    Returns
    -------
    corr_df_by_target : pd.DataFrame or dict
        For a single target: the correlation dataframe.
        For multiple targets: {target_col: corr_df}.

    significant_by_target : dict
        {target_col: [(feature, spearman_r), ...]}
    """

    if isinstance(target_col, str):
        target_cols = [target_col]
        scalar_target = True
    else:
        # Accept list/tuple/set/np.ndarray/pd.Series.
        target_cols = list(target_col)
        scalar_target = False

    if len(target_cols) == 0:
        raise ValueError("target_col must be a non-empty string or iterable of strings")

    exclude_cols = ['antibody_id', 'residue_number', 'n_total_rows', 'n_filtered_rows', 'n_beta_sheet_rows', 'n_exposed_rows']
    id_cols = {'structure_id', 'base', 'heavy', 'light', 'dataset', 'name', 'antibody_name'}

    corr_df_by_target = {}
    significant_by_target = {}

    for tcol in target_cols:
        merged_df_t = merged_df.copy()
        merged_df_t[tcol] = pd.to_numeric(merged_df_t[tcol], errors="coerce")

        n_before = len(merged_df_t)
        merged_df_t = merged_df_t.dropna(subset=[tcol])
        n_after = len(merged_df_t)
        if n_before > n_after:
            print(f"Dropped {n_before - n_after} rows with NaN/invalid target '{tcol}' (using {n_after} for correlations).")

        numeric_cols = merged_df_t.select_dtypes(include=[np.number]).columns.tolist()
        if len(numeric_cols) > 0:
            feature_cols = [
                col
                for col in numeric_cols
                if col not in exclude_cols
                and col not in id_cols
                and col not in target_cols
                and not str(col).startswith("target")
            ]
        else:
            feature_cols = [
                col
                for col in merged_df_t.columns
                if col not in exclude_cols
                and col not in id_cols
                and col not in target_cols
                and not str(col).startswith("target")
            ]

        correlations = []
        for col in feature_cols:
            data = merged_df_t[[tcol, col]].copy()
            data[tcol] = pd.to_numeric(data[tcol], errors='coerce')
            data[col] = pd.to_numeric(data[col], errors='coerce')
            data = data.dropna()
            if len(data) < 3:
                continue

            if normalize:
                target_min = data[tcol].min()
                target_max = data[tcol].max()
                target_range = target_max - target_min
                if target_range > 0:
                    target_norm = (data[tcol] - target_min) / target_range
                else:
                    target_norm = data[tcol]

                feature_min = data[col].min()
                feature_max = data[col].max()
                feature_range = feature_max - feature_min
                if feature_range > 0:
                    feature_norm = (data[col] - feature_min) / feature_range
                else:
                    feature_norm = data[col]

                spearman_r, spearman_p = spearmanr(target_norm, feature_norm)
            else:
                spearman_r, spearman_p = spearmanr(data[tcol], data[col])

            correlations.append({
                'feature': col,
                'spearman_r': spearman_r,
                'spearman_p': spearman_p,
                'n_samples': len(data)
            })

        corr_df = pd.DataFrame(correlations)
        if corr_df.empty:
            corr_df = pd.DataFrame(columns=['feature', 'spearman_r', 'spearman_p', 'spearman_p_adj', 'n_samples'])
            significant = corr_df.copy()
        else:
            if use_fdr:
                corr_df['spearman_p_adj'] = _bh_adjust(corr_df['spearman_p'].values)
                significant = corr_df[corr_df['spearman_p_adj'] < fdr_alpha].copy()
            else:
                corr_df['spearman_p_adj'] = corr_df['spearman_p'].values
                significant = corr_df[corr_df['spearman_p'] < p_threshold].copy()

            # NEW: apply correlation magnitude threshold
            if correlation_threshold is not None:
                significant = significant[
                    significant["spearman_r"].abs() >= correlation_threshold
                ].copy()

        print(f"[target={tcol}] Total features tested: {len(corr_df)}")
        print(f"[target={tcol}] Significant (raw p < {p_threshold}): {(corr_df['spearman_p'] < p_threshold).sum()}")
        if use_fdr:
            print(f"[target={tcol}] Significant after FDR correction (adj p < {fdr_alpha}): {len(significant)}")
        else:
            print(f"[target={tcol}] Significant (no FDR): {len(significant)}")
        if correlation_threshold is not None:
            print(
                f"[target={tcol}] After applying |rho| >= {correlation_threshold}: {len(significant)}"
            )
        if normalize:
            print(f"[target={tcol}] Note: Features were min-max normalized before correlation calculation")
        print(
            f"[target={tcol}] Top correlations by absolute Spearman r ({'FDR-significant only' if use_fdr else 'raw p < ' + str(p_threshold)}):"
        )

        if len(significant) > 0:
            sig = significant.copy()
            sig["spearman_r"] = pd.to_numeric(sig["spearman_r"], errors="coerce")
            sig = sig.dropna(subset=["spearman_r"])
            if len(sig) > 0:
                cols = ["feature", "spearman_r", "spearman_p", "spearman_p_adj"]
                print(sig.nlargest(10, "spearman_r", keep="all")[[c for c in cols if c in sig.columns]])
            else:
                print("(none)")
        else:
            print("(no significant correlations)")

        if make_plots and len(significant) > 0:
            n_plots = len(significant)
            n_cols = 3
            n_rows = (n_plots + n_cols - 1) // n_cols

            fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
            axes = np.atleast_1d(axes).flatten()

            for plot_idx, (idx, row) in enumerate(significant.iterrows()):
                ax = axes[plot_idx]
                data = merged_df_t[[tcol, row['feature']]].copy()
                data[tcol] = pd.to_numeric(data[tcol], errors='coerce')
                data[row['feature']] = pd.to_numeric(data[row['feature']], errors='coerce')
                data = data.dropna()

                if normalize:
                    target_min = data[tcol].min()
                    target_max = data[tcol].max()
                    target_range = target_max - target_min
                    if target_range > 0:
                        target_norm = (data[tcol] - target_min) / target_range
                    else:
                        target_norm = data[tcol]

                    feature_min = data[row['feature']].min()
                    feature_max = data[row['feature']].max()
                    feature_range = feature_max - feature_min
                    if feature_range > 0:
                        feature_norm = (data[row['feature']] - feature_min) / feature_range
                    else:
                        feature_norm = data[row['feature']]

                    ax.scatter(feature_norm, target_norm, alpha=0.6)

                    z = np.polyfit(feature_norm, target_norm, 1)
                    p = np.poly1d(z)
                    # ax.plot(feature_norm, p(feature_norm), "r--", alpha=0.8)
                else:
                    ax.scatter(data[row['feature']], data[tcol], alpha=0.6)

                    z = np.polyfit(data[row['feature']], data[tcol], 1)
                    p = np.poly1d(z)
                    # ax.plot(data[row['feature']], p(data[row['feature']]), "r--", alpha=0.8)

                ax.set_xlabel(row['feature'], fontsize=10)
                ax.set_ylabel(tcol, fontsize=10)
                p_label = 'p_adj' if use_fdr else 'p'
                ax.set_title(f"ρ={row['spearman_r']:.3f}, {p_label}={row['spearman_p_adj']:.3e}", fontsize=9)
                ax.grid(True, alpha=0.3)

            for idx in range(len(significant), len(axes)):
                axes[idx].axis('off')

            plt.tight_layout()
            plt.show()

        # Return list of (feature, spearman_r) tuples sorted by |spearman_r|.
        significant_tuples = []
        if len(significant) > 0:
            sig = significant.copy()
            sig["spearman_r"] = pd.to_numeric(sig["spearman_r"], errors="coerce")
            sig = sig.dropna(subset=["spearman_r"])
            if len(sig) > 0:
                sig = sig.assign(_abs_r=sig["spearman_r"].abs())
                sig = sig.sort_values("_abs_r", ascending=False).drop(columns=["_abs_r"])
                significant_tuples = list(
                    zip(
                        sig["feature"].astype(str).tolist(),
                        sig["spearman_r"].astype(float).tolist(),
                    )
                )

        corr_df_by_target[tcol] = corr_df
        significant_by_target[tcol] = significant_tuples

    if scalar_target:
        return corr_df_by_target[target_cols[0]], significant_by_target

    return corr_df_by_target, significant_by_target

In [96]:
def stability_selection_features_cv(
    merged_df,  # just one train split dataframe
    target_cols,
    n_subsamples: int = 100,
    sample_fraction: float = 0.5,
    model_type: str = "elasticnet",
    l1_ratio: float = 0.7,
    alpha: float = 0.01,
    coef_threshold: float = 1e-3,
    candidate_features=None,
    random_state: int = 42,
    verbose: bool = True,
):
    """
    Stability selection applied to multiple targets in one train split.

    Returns:
        freq_by_target: dict[target_col, pd.Series]
        selected_by_target: dict[target_col, list[str]]
    """
    from sklearn.linear_model import ElasticNet
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.preprocessing import StandardScaler
    import numpy as np
    import pandas as pd

    if isinstance(target_cols, str):
        target_cols = [target_cols]

    freq_by_target = {}
    selected_by_target = {}

    for target_col in target_cols:
        exclude_cols = {target_col, "target", "viscosity", "target_viscosity",
                        "structure_id", "base", "heavy", "light", "name", "dataset", "index"}

        if candidate_features is not None:
            feat_candidates = [f for f in candidate_features if f in merged_df.columns and f not in exclude_cols]
        else:
            numeric_cols = merged_df.select_dtypes(include=[np.number]).columns
            feat_candidates = [c for c in numeric_cols if c not in exclude_cols]

        if len(feat_candidates) == 0:
            freq_by_target[target_col] = pd.Series(dtype=float)
            selected_by_target[target_col] = []
            continue

        X = merged_df[feat_candidates].apply(pd.to_numeric, errors="coerce").fillna(0).values
        y = pd.to_numeric(merged_df[target_col], errors="coerce").values
        mask = ~np.isnan(y)
        X = X[mask]
        y = y[mask]

        if len(y) < 4:
            freq_by_target[target_col] = pd.Series(dtype=float)
            selected_by_target[target_col] = []
            continue

        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        rng = np.random.default_rng(random_state)
        n_feats = len(feat_candidates)
        feature_counts = np.zeros(n_feats)
        subsize = max(2, int(len(y) * sample_fraction))

        for i in range(n_subsamples):
            idx = rng.choice(len(y), size=subsize, replace=False)
            X_sub = X_scaled[idx]
            y_sub = y[idx]

            if model_type.lower() == "elasticnet":
                model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=5000,
                                   random_state=rng.integers(0, 2**31))
                model.fit(X_sub, y_sub)
                non_zero = np.abs(model.coef_) > coef_threshold

            elif model_type.lower() in ("randomforest", "random_forest", "rf"):
                from sklearn.ensemble import RandomForestRegressor
                model = RandomForestRegressor(n_estimators=100, max_features="sqrt",
                                              random_state=rng.integers(0, 2**31))
                model.fit(X_sub, y_sub)
                non_zero = model.feature_importances_ > coef_threshold
            else:
                raise ValueError("model_type must be 'elasticnet' or 'randomforest'")

            feature_counts[non_zero] += 1
            if verbose and (i + 1) % 20 == 0:
                print(f"Stability selection target={target_col}: {i + 1}/{n_subsamples} subsamples")

        freq = pd.Series(feature_counts / n_subsamples, index=feat_candidates)
        freq_by_target[target_col] = freq

        # Elbow selection (Kneedle)
        freq_sorted = freq.sort_values(ascending=False)
        x = np.arange(len(freq_sorted))
        y_vals = freq_sorted.values
        start, end = np.array([x[0], y_vals[0]]), np.array([x[-1], y_vals[-1]])
        line_vec = end - start
        norm = np.linalg.norm(line_vec)
        if norm == 0:
            elbow_idx = 0
        else:
            line_dir = line_vec / norm
            pts = np.stack([x, y_vals], axis=1)
            diffs = pts - start
            proj_lengths = diffs @ line_dir
            proj_points = np.outer(proj_lengths, line_dir)
            orthogonal = diffs - proj_points
            dists = np.linalg.norm(orthogonal, axis=1)
            elbow_idx = int(np.argmax(dists))

        k_feats = max(1, elbow_idx + 1)
        selected_by_target[target_col] = freq_sorted.iloc[:k_feats].index.tolist()

    return freq_by_target, selected_by_target

In [197]:
from sklearn.linear_model import ElasticNet, Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr, pearsonr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def fit_and_compare_models(
    merged_train,
    merged_test,
    target_col,
    feature_list,
    enet_alpha,              # alpha chosen outside (e.g. via CV)
    random_state=42,
    enet_l1_ratio=0.5,
    max_feature_fraction=0.2,
    make_plots=False,
):
    """
    Fit ElasticNet, Ridge, LinearRegression, and RandomForest on TRAIN data,
    evaluate on TEST data.

    - ElasticNet with provided enet_alpha is used on train set to select features:
      max number of features = n_train_samples * max_feature_fraction (top by |coef|).
    - All models are then fit on the selected feature set using train data.
    - Metrics and plots are computed on the test set.

    Returns
    -------
    selected_features : list
        Features kept after ElasticNet-based selection (from train data).
    results_df : pd.DataFrame
        Per-model metrics on the test data:
        ['model', 'pearson_r', 'spearman_r', 'spearman_p', 'n_features', 'n_train', 'n_test'].
    """
    # Ensure we don't accidentally include the target itself as a feature (no leakage)
    feature_list = [f for f in feature_list if f not in ("target", target_col, "target_viscosity", "viscosity") and not str(f).startswith("target")]

    X_train = merged_train[feature_list].copy()
    X_test = merged_test[feature_list].copy()

    for c in feature_list:
        X_train[c] = pd.to_numeric(X_train[c], errors="coerce")
        X_test[c] = pd.to_numeric(X_test[c], errors="coerce")

    y_train = pd.to_numeric(merged_train[target_col], errors="coerce")
    y_test = pd.to_numeric(merged_test[target_col], errors="coerce")

    train_mask = y_train.notna() & X_train.notna().all(axis=1)
    test_mask = y_test.notna() & X_test.notna().all(axis=1)

    X_train = X_train.loc[train_mask].values
    y_train = y_train.loc[train_mask].values
    X_test = X_test.loc[test_mask].values
    y_test = y_test.loc[test_mask].values

    n_train = len(y_train)
    n_test = len(y_test)

    if n_train < 3 or n_test < 1:
        raise ValueError("Not enough samples to fit/evaluate models.")

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # ElasticNet-based feature selection using externally provided alpha (train only)
    enet = ElasticNet(
        alpha=enet_alpha,
        l1_ratio=enet_l1_ratio,
        random_state=random_state,
    )
    enet.fit(X_train_scaled, y_train)

    n_keep = max(1, int(n_train * max_feature_fraction))
    n_keep = min(n_keep, len(feature_list))
    order = np.argsort(np.abs(enet.coef_))[::-1]
    top_indices = order[:n_keep]
    selected_features = [feature_list[i] for i in top_indices]

    X_train_sel = X_train_scaled[:, top_indices]
    X_test_sel = X_test_scaled[:, top_indices]

    # Models: fit on train, evaluate on test
    models = {
        "ElasticNet": ElasticNet(
            alpha=enet_alpha,
            l1_ratio=enet_l1_ratio,
            random_state=random_state,
        ),
        "Ridge": Ridge(alpha=enet_alpha, random_state=random_state),
        "Linear": LinearRegression(),
        "RandomForest": RandomForestRegressor(
            n_estimators=100,
            max_features="sqrt",
            random_state=random_state,
        ),
    }

    results = []
    preds = {}  # for plotting
    for name, model in models.items():
        model.fit(X_train_sel, y_train)
        y_pred = model.predict(X_test_sel)
        preds[name] = y_pred
        pr_r, pr_p = pearsonr(y_test, y_pred)
        sp_r, sp_p = spearmanr(y_test, y_pred)
        results.append(
            {
                "model": name,
                "pearson_r": float(pr_r) if not np.isnan(pr_r) else np.nan,
                "spearman_r": float(sp_r) if not np.isnan(sp_r) else np.nan,
                "spearman_p": float(sp_p) if not np.isnan(sp_p) else np.nan,
                "n_features": len(selected_features),
                "n_train": n_train,
                "n_test": n_test,
            }
        )

    results_df = pd.DataFrame(results)

    # --- Plots (on test set) ---
    # 1) Observed vs predicted (one panel per model)
    if make_plots:
        n_models = len(preds)
        fig, axes = plt.subplots(1, n_models, figsize=(4 * n_models, 4))
        axes = np.atleast_1d(axes)
        for ax, (name, y_pred) in zip(axes, preds.items()):
            ax.scatter(y_test, y_pred, alpha=0.7, edgecolors="k", linewidths=0.5)
            lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
            ax.plot(lims, lims, "r--", label="y = ŷ")
            ax.set_xlabel("Observed (test)")
            ax.set_ylabel("Predicted (test)")
            ax.set_title(f"{name} (train→test, n_test={n_test})")
            ax.legend(loc="upper left", fontsize=8)
            ax.set_aspect("equal", adjustable="box")
            ax.grid(True, alpha=0.3)
        plt.suptitle(
            f"Test predictions (train fit, no internal CV)\nX_test shape: ({n_test}, {len(selected_features)})",
            fontsize=11,
        )
        plt.tight_layout()
        plt.show()

        # 2) Bar chart: R² and Spearman r per model (on test)
        fig, ax = plt.subplots(figsize=(6, 4))
        x = np.arange(len(results_df))
        w = 0.35
        ax.bar(x - w / 2, results_df["pearson_r"], w, label="Pearson r", color="steelblue", alpha=0.8)
        ax.bar(x + w / 2, results_df["spearman_r"], w, label="Spearman r", color="coral", alpha=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(results_df["model"])
        ax.set_ylabel("Score (test)")
        ax.legend()
        ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.3, axis="y")
        plt.tight_layout()
        plt.show()

    return selected_features, results_df

In [109]:
from sklearn.feature_selection import mutual_info_regression

def reduce_correlated_features(
    split_train_df: "pd.DataFrame",
    target_cols=None,
    feature_set=None,
    correlation_threshold: float = 0.8,
    importance_metric: str = "spearman",  # "spearman" or "mutual_info"
):
    """Reduce highly correlated features within ONE train split.

    This is done per target column, returning a dict:
    ``{target_col: kept_features}``.

    Parameters
    ----------
    split_train_df:
        Training split dataframe.
    target_cols:
        Target column name(s). If None, tries to infer from ``feature_set``
        keys (dict) or from a global ``target_cols`` variable.
    feature_set:
        Optional feature set(s) to restrict the pruning.
        - list[str]: applies to all targets
        - dict[target_col, list[str]]: per-target restriction
        - None: use all numeric columns (excluding target/ID-like columns)

    Returns
    -------
    reduced_features_by_target : dict[str, list[str]]
    """

    import pandas as pd
    import numpy as np
    from scipy.stats import spearmanr
    from sklearn.feature_selection import mutual_info_regression

    importance_metric = importance_metric.lower()
    if importance_metric not in ("spearman", "mutual_info", "pearson"):
        raise ValueError("importance_metric must be 'spearman' or 'mutual_info' or 'pearson'")

    # Normalize target_cols input / inference.
    if target_cols is None:
        if isinstance(feature_set, dict) and len(feature_set) > 0:
            target_cols = list(feature_set.keys())
        elif "target_cols" in globals() and globals().get("target_cols") is not None:
            target_cols = list(globals().get("target_cols"))
        else:
            raise ValueError("target_cols must be provided (or inferred from feature_set keys).")

    if isinstance(target_cols, str):
        target_cols = [target_cols]
    else:
        target_cols = list(target_cols)

    target_cols = [str(t) for t in target_cols]
    if not target_cols:
        raise ValueError("target_cols must be non-empty")

    # Global exclude list to avoid pruning IDs/metadata and avoid selecting targets as features.
    global_exclude_cols = {
        "antibody_id",
        "structure_id",
        "residue_number",
        "n_total_rows",
        "n_filtered_rows",
        "n_beta_sheet_rows",
        "n_exposed_rows",
        "base",
        "heavy",
        "light",
        "name",
        "dataset",
        "antibody_name",
        "index",
        "target",
        "viscosity",
        "target_viscosity",
    }

    # Interpret feature_set as a per-target restriction.
    feature_set_by_target = {}
    if feature_set is None:
        feature_set_by_target = {t: None for t in target_cols}
    elif isinstance(feature_set, dict):
        for t in target_cols:
            feature_set_by_target[t] = feature_set.get(t, None)
    else:
        # Treat list/iterable as one shared feature set.
        shared = list(feature_set)
        feature_set_by_target = {t: shared for t in target_cols}

    reduced_features_by_target = {}

    for target_col in target_cols:
        # Determine candidate features for this target.
        if feature_set_by_target.get(target_col) is None:
            numeric_cols = split_train_df.select_dtypes(include=[np.number]).columns.tolist()
            sig_features = [
                f
                for f in numeric_cols
                if f not in global_exclude_cols
                and f not in target_cols
                and not str(f).startswith("target")
            ]
        else:
            fs = set(feature_set_by_target[target_col])
            sig_features = [
                f
                for f in fs
                if f in split_train_df.columns
                and f not in global_exclude_cols
                and f not in target_cols
                and not str(f).startswith("target")
            ]

        if len(sig_features) < 2:
            reduced_features_by_target[target_col] = sig_features
            continue

        # Prepare numeric data including the target.
        data = split_train_df[sig_features + [target_col]].copy()
        for c in sig_features:
            data[c] = pd.to_numeric(data[c], errors="coerce")
        data[target_col] = pd.to_numeric(data[target_col], errors="coerce")
        data = data.dropna(how="all")
        if len(data) < 3:
            reduced_features_by_target[target_col] = sig_features
            continue

        # Target importance per feature (to decide which member of a correlated pair to keep).
        target_score = {}
        for f in sig_features:
            sub = data[[f, target_col]].dropna()
            if len(sub) < 3:
                target_score[f] = 0.0
                continue

            if importance_metric == "spearman":
                r, _ = spearmanr(sub[f], sub[target_col])
                score = float(r) if not np.isnan(r) else 0.0
            else:  # mutual_info
                X_f = sub[[f]].values
                y_t = sub[target_col].values
                mi = mutual_info_regression(X_f, y_t, discrete_features=False)
                score = float(mi[0]) if len(mi) > 0 and not np.isnan(mi[0]) else 0.0

            target_score[f] = score

        dropped = set()

        # Pairwise feature–feature Pearson on same (train) data.
        for i, f1 in enumerate(sig_features):
            if f1 in dropped:
                continue
            for f2 in sig_features[i + 1 :]:
                if f2 in dropped:
                    continue

                sub = data[[f1, f2]].dropna()
                if len(sub) < 3:
                    continue

                r, _ = pearsonr(sub[f1], sub[f2])
                if np.isnan(r) or abs(r) < correlation_threshold:
                    continue

                # Keep the feature with higher importance vs the target.
                s1 = abs(target_score.get(f1, 0.0))
                s2 = abs(target_score.get(f2, 0.0))
                if s1 >= s2:
                    remove = f2
                else:
                    remove = f1

                dropped.add(remove)

        kept_features = [f for f in sig_features if f not in dropped]
        reduced_features_by_target[target_col] = kept_features

    return reduced_features_by_target

# Load data

In [153]:
# Experimental datasets + matching descriptor result folders
df_ab21 = load_and_merge(
    exp_csv_path=Path("../data/ab21.csv"),
    results_dir_path=Path("../ab21_results"),
    base="ab21",
)

df_pdgf38 = load_and_merge(
    exp_csv_path=Path("../data/pdgf38.csv"),
    results_dir_path=Path("../pdgf38_results"),
    base="pdgf38",
)

df_garbinski2023_tm1 = load_and_merge(
    exp_csv_path=Path("../data/garbinski2023_tm1.csv"),
    results_dir_path=Path("../garbinski2023_results"),
    base="garbinski2023",
)

# Keep legacy `df` as the garbinski merged dataframe
df = df_garbinski2023_tm1

df_ginkgo = load_and_merge(
    exp_csv_path=Path("../data/ginkgo.csv"),
    results_dir_path=Path("../GINKGO_results"),
    base="GINKGO",
)

df_hutchinson2023enhancement_top200tm1_igg = load_and_merge(
    exp_csv_path=Path("../data/hutchinson2023enhancement_top200tm1_igg.csv"),
    results_dir_path=Path("../hutchinson2023enhancement_results"),
    base="hutchinson2023enhancement",
)

df_jain2017biophysical = load_and_merge(
    exp_csv_path=Path("../data/jain2017biophysical.csv"),
    results_dir_path=Path("../jain2017biophysical_results"),
    base="jain2017biophysical",
)

df_jain2023identifying = load_and_merge(
    exp_csv_path=Path("../data/jain2023identifying.csv"),
    results_dir_path=Path("../jain2023identifying_results"),
    base="jain2023identifying",
)

df_jain2024assessment = load_and_merge(
    exp_csv_path=Path("../data/jain2024assessment.csv"),
    results_dir_path=Path("../jain2024assessment_results"),
    base="jain2024assessment",
)

df_jetha2019homology_RT = load_and_merge(
    exp_csv_path=Path("../data/jetha2019homology_RT.csv"),
    results_dir_path=Path("../jetha2019homology_results"),
    base="jetha2019homology",
)

df_kraft2019herapin_relrt = load_and_merge(
    exp_csv_path=Path("../data/kraft2019herapin_relrt.csv"),
    results_dir_path=Path("../kraft2019herapin_results"),
    base="kraft2019herapin",
)


Merged experimental ab21.csv with results 'ab21': 21 rows
Merged experimental pdgf38.csv with results 'pdgf38': 38 rows
Merged experimental garbinski2023_tm1.csv with results 'garbinski2023': 86 rows
Merged experimental ginkgo.csv with results 'GINKGO': 246 rows
Merged experimental hutchinson2023enhancement_top200tm1_igg.csv with results 'hutchinson2023enhancement': 192 rows
Merged experimental jain2017biophysical.csv with results 'jain2017biophysical': 137 rows
Merged experimental jain2023identifying.csv with results 'jain2023identifying': 115 rows
Merged experimental jain2024assessment.csv with results 'jain2024assessment': 43 rows
Merged experimental jetha2019homology_RT.csv with results 'jetha2019homology': 97 rows
Merged experimental kraft2019herapin_relrt.csv with results 'kraft2019herapin': 128 rows


# Data analytics

In [103]:
feature_cols = df_kraft2019herapin_relrt.columns.values.tolist()
feature_cols.remove("name")
feature_cols.remove("heavy")
feature_cols.remove("light")
feature_cols.remove("target_fitness")
feature_cols.remove("base")

In [308]:
_, _ = calculate_correlations_and_plot(df_pdgf38, ['target_viscosity'], p_threshold=0.05, fdr_alpha=0.05, use_fdr=False, normalize=False, make_plots=False, correlation_threshold=0.0)

[target=target_viscosity] Total features tested: 188
[target=target_viscosity] Significant (raw p < 0.05): 67
[target=target_viscosity] Significant (no FDR): 67
[target=target_viscosity] After applying |rho| >= 0.0: 67
[target=target_viscosity] Top correlations by absolute Spearman r (raw p < 0.05):
                                               feature  spearman_r  \
118  total_side_rel_sums_polar_exposed_total_side_r...    0.750055   
54           density_metrics_avg_negative_cdr_over_cdr    0.740575   
183         sequence_motives_side_asa_sum_motif_AspAsp    0.720079   
50                       density_metrics_avg_polar_all    0.712558   
116       total_side_rel_sums_polar_total_side_rel_sum    0.712558   
95          charge_metrics_weighted_scm_score_by_pH_10    0.693806   
73       topology_metrics_relative_contact_order_light    0.688548   
110  total_side_rel_sums_negative_exposed_total_sid...    0.684870   
51                    density_metrics_avg_negative_all    0.680784   

In [ ]:
'target_ACSINS', 'target_CIC',
       'target_CSSINS', 'target_Fab_pI', 'target_HIC',
       'target_Herapin_RT', 'target_PSR', 'target_SEC', 'target_Tm',
       'target_cIEF',

In [313]:
all_sign_feats = []
for dataset in [df_jain2024assessment, df_jain2023identifying, df_jain2017biophysical, df_kraft2019herapin_relrt, df_garbinski2023_tm1, df_ab21, df_pdgf38, df_ginkgo, df_hutchinson2023enhancement_top200tm1_igg, df_jetha2019homology_RT]:
    target_cols = [col for col in dataset.columns.values.tolist() if col.startswith('target_')]
    _, features = calculate_correlations_and_plot(dataset, target_cols, p_threshold=0.05, fdr_alpha=0.05, use_fdr=False, normalize=False, make_plots=False, correlation_threshold=0.0)
    all_sign_feats.append(features)

[target=target_ACSINS] Total features tested: 188
[target=target_ACSINS] Significant (raw p < 0.05): 11
[target=target_ACSINS] Significant (no FDR): 11
[target=target_ACSINS] After applying |rho| >= 0.0: 11
[target=target_ACSINS] Top correlations by absolute Spearman r (raw p < 0.05):
                                               feature  spearman_r  \
106  total_side_rel_sums_aromatic_exposed_total_sid...    0.354665   
43                    density_metrics_avg_aromatic_all    0.325728   
89           charge_metrics_weighted_scm_score_by_pH_4    0.321515   
104    total_side_rel_sums_aromatic_total_side_rel_sum    0.313795   
73       topology_metrics_relative_contact_order_light   -0.322995   
30                       h_bonds_metrics_avg_hbond_cdr   -0.341950   
159               sequence_motives_n_motif_AspAsp_CDRs   -0.361310   
133      other_sasa_metrics_avg_total_side_rel_exposed   -0.370783   
185         sequence_motives_side_asa_sum_motif_AspHis   -0.431903   
140           

In [314]:
all_sign_feats

[{'target_ACSINS': [('sequence_motives_n_motif_AspHis_CDRs',
    -0.43329886270727985),
   ('sequence_motives_n_motif_AspHis', -0.43329886270727985),
   ('sequence_motives_side_asa_sum_motif_AspHis', -0.4319034847792402),
   ('other_sasa_metrics_avg_total_side_rel_exposed', -0.3707832749150117),
   ('sequence_motives_n_motif_AspAsp_CDRs', -0.36131043334522467),
   ('total_side_rel_sums_aromatic_exposed_total_side_rel_sum',
    0.354665151756135),
   ('h_bonds_metrics_avg_hbond_cdr', -0.34194988383145924),
   ('density_metrics_avg_aromatic_all', 0.3257275087966904),
   ('topology_metrics_relative_contact_order_light', -0.322995040145378),
   ('charge_metrics_weighted_scm_score_by_pH_4', 0.3215146318096589),
   ('total_side_rel_sums_aromatic_total_side_rel_sum', 0.3137946477125343)],
  'target_CIC': [('topology_metrics_relative_contact_order_light',
    -0.5205190645101735),
   ('sequence_motives_n_motif_AspAsp_CDRs', -0.39778671383727643),
   ('cluster_metrics_aromatic_exposed_cluster_n

In [311]:
all_sign_feats = []
for dataset in [df_jain2024assessment, df_jain2023identifying, df_jain2017biophysical, df_kraft2019herapin_relrt, df_garbinski2023_tm1, df_ab21, df_pdgf38, df_ginkgo, df_hutchinson2023enhancement_top200tm1_igg, df_jetha2019homology_RT]:
    target_cols = [col for col in dataset.columns.values.tolist() if col.startswith('target_')]
    _, features = calculate_correlations_and_plot(dataset, target_cols, p_threshold=0.05, fdr_alpha=0.05, use_fdr=False, normalize=False, make_plots=False, correlation_threshold=0.0)
    for key in features.keys():
        for el in features[key]:
            all_sign_feats.append(el[0])

[target=target_ACSINS] Total features tested: 188
[target=target_ACSINS] Significant (raw p < 0.05): 11
[target=target_ACSINS] Significant (no FDR): 11
[target=target_ACSINS] After applying |rho| >= 0.0: 11
[target=target_ACSINS] Top correlations by absolute Spearman r (raw p < 0.05):
                                               feature  spearman_r  \
106  total_side_rel_sums_aromatic_exposed_total_sid...    0.354665   
43                    density_metrics_avg_aromatic_all    0.325728   
89           charge_metrics_weighted_scm_score_by_pH_4    0.321515   
104    total_side_rel_sums_aromatic_total_side_rel_sum    0.313795   
73       topology_metrics_relative_contact_order_light   -0.322995   
30                       h_bonds_metrics_avg_hbond_cdr   -0.341950   
159               sequence_motives_n_motif_AspAsp_CDRs   -0.361310   
133      other_sasa_metrics_avg_total_side_rel_exposed   -0.370783   
185         sequence_motives_side_asa_sum_motif_AspHis   -0.431903   
140           

In [306]:
from collections import Counter

counts = Counter(all_sign_feats)
counts


Counter({'charge_metrics_net_charge_by_pH_7': 24,
         'charge_metrics_net_charge_by_pH_8': 24,
         'charge_metrics_net_charge_by_pH_10': 24,
         'charge_metrics_net_charge_by_pH_6': 24,
         'total_side_rel_sums_positive_total_side_rel_sum': 24,
         'cluster_metrics_pnc_cdr_vicinity': 24,
         'charge_metrics_weighted_scm_score_by_pH_4': 23,
         'charge_metrics_heavy_charge_pH7': 23,
         'charge_metrics_net_charge_by_pH_9': 23,
         'charge_metrics_sap_neg_charge_score': 23,
         'density_metrics_avg_positive_all': 23,
         'density_metrics_avg_negative_cdr_over_cdr': 23,
         'charge_metrics_weighted_scm_score_by_pH_7': 22,
         'charge_metrics_weighted_scm_score_by_pH_6': 22,
         'total_side_rel_sums_positive_exposed_total_side_rel_sum': 22,
         'density_metrics_avg_aromatic_all': 21,
         'total_side_rel_sums_aromatic_total_side_rel_sum': 21,
         'charge_metrics_net_charge_by_pH_5': 21,
         'charge_met

# Build model

In [276]:
df_our = df_ginkgo
target_cols = [
       'target_SEC_Monomer', 'target_SMAC',
       'target_HIC', 'target_HAC', 'target_PR_CHO', 'target_PR_Ova',
       'target_AC_SINS_pH6_0', 'target_AC_SINS_pH7_4', 'target_Tonset',
       'target_Tm1', 'target_Tm2'
       ]

# _, _ = calculate_correlations_and_plot(df_our, 'target_CSIBLI', p_threshold=0.05, fdr_alpha=0.05, use_fdr=True, normalize=False, make_plots=True, correlation_threshold=0.0)

In [ ]:
# Drop rows where ANY target is missing (strict mode)
data = df_our.dropna(subset=target_cols).copy()

n_splits = 5
random_state = 42

N = len(data)
if n_splits < 2 or n_splits > N:
    raise ValueError(f"n_splits must be between 2 and N={N}.")

base_size = N // n_splits
remainder = N % n_splits
segment_sizes = [base_size + (1 if i < remainder else 0) for i in range(n_splits)]
boundaries = np.cumsum([0] + segment_sizes)

rng = np.random.default_rng(random_state)
idx = np.arange(N)
rng.shuffle(idx)

train_splits = []  # [[(df_k, target1), (df_k, target2), ...]]
test_splits = []

for i in range(n_splits):
    test_start, test_end = boundaries[i], boundaries[i + 1]
    test_idx = idx[test_start:test_end]
    train_idx = np.concatenate([idx[:test_start], idx[test_end:]])

    train_df_k = data.iloc[train_idx].copy()
    test_df_k = data.iloc[test_idx].copy()

    # 🔥 create per-target pairs
    train_splits.append([(train_df_k.copy(), t) for t in target_cols])
    test_splits.append([(test_df_k.copy(), t) for t in target_cols])

# -------------------------------
# Low-variance filtering
# -------------------------------

low_variance_relative_std_threshold = 0.05
low_variance_epsilon = 1e-8

exclude_cols = {
    "base",
    "heavy",
    "light",
    "name",
    "index"
}

features_lowvar = []
corr_results_per_split = []
features_significant_corrs = []
features_significant_corrs_reduced_per_split = []
features_significant_corrs_per_split = []
freq_ss_per_split = []
features_stability_per_split = []
features_stability_reduced_per_split = []


for k in range(n_splits):
    # Use first target just to compute feature filtering (same df anyway)
    train_df_k, _ = train_splits[k][0]
    test_df_k, _ = test_splits[k][0]

    candidate_features = [
        c
        for c in train_df_k.columns
        if c not in exclude_cols
        and c not in target_cols
        and not str(c).startswith("target")
    ]

    kept_k, removed_k, rel_std_k = remove_low_variance_features(
        X=train_df_k,
        feature_cols=candidate_features,
        relative_std_threshold=low_variance_relative_std_threshold,
        epsilon=low_variance_epsilon,
    )

    if len(kept_k) == 0:
        finite_feats = rel_std_k.index[np.isfinite(rel_std_k.values)].tolist()
        kept_k = finite_feats if len(finite_feats) > 0 else candidate_features[:1]

    kept_k_set = set(kept_k)
    removed_k = [c for c in candidate_features if c not in kept_k_set]

    # Apply to base dfs
    train_df_k = train_df_k.drop(columns=removed_k, errors="ignore").copy()
    test_df_k = test_df_k.drop(columns=removed_k, errors="ignore").copy()

    # 🔥 rebuild split with filtered dfs
    train_splits[k] = [(train_df_k.copy(), t) for t in target_cols]
    test_splits[k] = [(test_df_k.copy(), t) for t in target_cols]

    features_lowvar.append(kept_k)

    # ----------------------------------
    # Correlation analysis per split
    # ----------------------------------

    # Run on TRAIN ONLY (important to avoid leakage)
    corr_df_k, significant_k = calculate_correlations_and_plot(
        merged_df=train_df_k,
        target_col=target_cols,
        p_threshold=0.05,
        fdr_alpha=0.05,
        use_fdr=True,
        normalize=False,
        make_plots=False,
        correlation_threshold=0,  # or your chosen value
    )

    # Store results
    corr_results_per_split.append(corr_df_k)
    features_significant_corrs_per_split.append(significant_k)

    # After calculate_correlations_and_plot
    sig_features_for_split = {t: [f for f, _ in features_significant_corrs_per_split[k][t]] for t in target_cols}

    features_significant_corrs_reduced = reduce_correlated_features(
        split_train_df=train_df_k,
        target_cols=target_cols,
        feature_set=sig_features_for_split,
        correlation_threshold=0.8,
        importance_metric="spearman",
    )
    features_significant_corrs_reduced_per_split.append(features_significant_corrs_reduced)


    freq_k, selected_k = stability_selection_features_cv(
        merged_df=train_df_k,
        target_cols=target_cols,
        n_subsamples=100,
        sample_fraction=0.5,
        model_type="elasticnet",
        l1_ratio=0.7,
        alpha=0.01,
        coef_threshold=1e-3,
        candidate_features=None,
        random_state=42,
        verbose=True,
    )
    freq_ss_per_split.append(freq_k)
    features_stability_per_split.append(selected_k)

    # Reduce correlated features for stability-selected features
    features_stability_reduced = reduce_correlated_features(
        split_train_df=train_df_k,
        target_cols=target_cols,
        feature_set=selected_k,
        correlation_threshold=0.8,
        importance_metric="spearman",
    )
    features_stability_reduced_per_split.append(features_stability_reduced)

print(
    "Low-variance kept feature counts per split:",
    [len(x) for x in features_lowvar],
)

print(
    f"Created {n_splits} splits: "
    f"test sizes {segment_sizes}, train sizes {[N - s for s in segment_sizes]}."
)

In [ ]:
for target_col in target_cols:
    results_per_split = []
    features = features_stability_reduced_per_split
    for split_idx in range(n_splits):
        train_df, _ = train_splits[split_idx][0]
        test_df, _ = test_splits[split_idx][0]

        # Suppose you already have final features for this target in this split:
        features_for_target = features[split_idx][target_col]

        # Choose alpha (from prior CV, e.g. 0.01)
        enet_alpha = 0.01

        selected_features, results_df = fit_and_compare_models(
            merged_train=train_df,
            merged_test=test_df,
            target_col=target_col,
            feature_list=features_for_target,
            enet_alpha=enet_alpha,
            enet_l1_ratio=0.7,          # optional
            max_feature_fraction=0.15,
            make_plots=False
        )

        results_per_split.append(results_df)

    results_agg = pd.concat(results_per_split, ignore_index=True)
    summary = results_agg.groupby("model").agg({"pearson_r": "mean", "spearman_r": "mean"}).reset_index()
    print("Mean test metrics across splits:")
    print(target_col)
    print(summary)

# Propermab

In [280]:
df = pd.read_csv("../GINKGO_propermab/features.csv")
exp_df = pd.read_csv("../data/ginkgo.csv")
exp_df['name'] = exp_df['name'].astype(str)
df['name'] = df['pdb_file'].str.split("/").str[-1].str.split('.').apply(lambda x: x[0])
df = df.drop(columns=['pdb_file'])
df = df.merge(exp_df, on='name', how='inner')

In [ ]:
# _, _ = calculate_correlations_and_plot(df, 'target_FvLysM2', p_threshold=0.05, fdr_alpha=0.05, use_fdr=True, normalize=False, make_plots=True, correlation_threshold=0.0)

In [ ]:
# Drop rows where ANY target is missing (strict mode)
data = df.dropna(subset=target_cols).copy()

n_splits = 5
random_state = 42

N = len(data)
if n_splits < 2 or n_splits > N:
    raise ValueError(f"n_splits must be between 2 and N={N}.")

base_size = N // n_splits
remainder = N % n_splits
segment_sizes = [base_size + (1 if i < remainder else 0) for i in range(n_splits)]
boundaries = np.cumsum([0] + segment_sizes)

rng = np.random.default_rng(random_state)
idx = np.arange(N)
rng.shuffle(idx)

train_splits = []  # [[(df_k, target1), (df_k, target2), ...]]
test_splits = []

for i in range(n_splits):
    test_start, test_end = boundaries[i], boundaries[i + 1]
    test_idx = idx[test_start:test_end]
    train_idx = np.concatenate([idx[:test_start], idx[test_end:]])

    train_df_k = data.iloc[train_idx].copy()
    test_df_k = data.iloc[test_idx].copy()

    # 🔥 create per-target pairs
    train_splits.append([(train_df_k.copy(), t) for t in target_cols])
    test_splits.append([(test_df_k.copy(), t) for t in target_cols])

# -------------------------------
# Low-variance filtering
# -------------------------------

low_variance_relative_std_threshold = 0.05
low_variance_epsilon = 1e-8

exclude_cols = {
    "base",
    "heavy",
    "light",
    "name",
    "index"
}

features_lowvar = []
corr_results_per_split = []
features_significant_corrs = []
features_significant_corrs_reduced_per_split = []
features_significant_corrs_per_split = []
freq_ss_per_split = []
features_stability_per_split = []
features_stability_reduced_per_split = []


for k in range(n_splits):
    # Use first target just to compute feature filtering (same df anyway)
    train_df_k, _ = train_splits[k][0]
    test_df_k, _ = test_splits[k][0]

    candidate_features = [
        c
        for c in train_df_k.columns
        if c not in exclude_cols
        and c not in target_cols
        and not str(c).startswith("target")
    ]

    kept_k, removed_k, rel_std_k = remove_low_variance_features(
        X=train_df_k,
        feature_cols=candidate_features,
        relative_std_threshold=low_variance_relative_std_threshold,
        epsilon=low_variance_epsilon,
    )

    if len(kept_k) == 0:
        finite_feats = rel_std_k.index[np.isfinite(rel_std_k.values)].tolist()
        kept_k = finite_feats if len(finite_feats) > 0 else candidate_features[:1]

    kept_k_set = set(kept_k)
    removed_k = [c for c in candidate_features if c not in kept_k_set]

    # Apply to base dfs
    train_df_k = train_df_k.drop(columns=removed_k, errors="ignore").copy()
    test_df_k = test_df_k.drop(columns=removed_k, errors="ignore").copy()

    # 🔥 rebuild split with filtered dfs
    train_splits[k] = [(train_df_k.copy(), t) for t in target_cols]
    test_splits[k] = [(test_df_k.copy(), t) for t in target_cols]

    features_lowvar.append(kept_k)

    # ----------------------------------
    # Correlation analysis per split
    # ----------------------------------

    # Run on TRAIN ONLY (important to avoid leakage)
    corr_df_k, significant_k = calculate_correlations_and_plot(
        merged_df=train_df_k,
        target_col=target_cols,
        p_threshold=0.05,
        fdr_alpha=0.05,
        use_fdr=True,
        normalize=False,
        make_plots=False,
        correlation_threshold=0.4,  # or your chosen value
    )

    # Store results
    corr_results_per_split.append(corr_df_k)
    features_significant_corrs_per_split.append(significant_k)

    # After calculate_correlations_and_plot
    sig_features_for_split = {t: [f for f, _ in features_significant_corrs_per_split[k][t]] for t in target_cols}

    features_significant_corrs_reduced = reduce_correlated_features(
        split_train_df=train_df_k,
        target_cols=target_cols,
        feature_set=sig_features_for_split,
        correlation_threshold=0.8,
        importance_metric="spearman",
    )
    features_significant_corrs_reduced_per_split.append(features_significant_corrs_reduced)


    freq_k, selected_k = stability_selection_features_cv(
        merged_df=train_df_k,
        target_cols=target_cols,
        n_subsamples=100,
        sample_fraction=0.5,
        model_type="elasticnet",
        l1_ratio=0.7,
        alpha=0.01,
        coef_threshold=1e-3,
        candidate_features=None,
        random_state=42,
        verbose=True,
    )
    freq_ss_per_split.append(freq_k)
    features_stability_per_split.append(selected_k)

    # Reduce correlated features for stability-selected features
    features_stability_reduced = reduce_correlated_features(
        split_train_df=train_df_k,
        target_cols=target_cols,
        feature_set=selected_k,
        correlation_threshold=0.8,
        importance_metric="spearman",
    )
    features_stability_reduced_per_split.append(features_stability_reduced)

print(
    "Low-variance kept feature counts per split:",
    [len(x) for x in features_lowvar],
)

print(
    f"Created {n_splits} splits: "
    f"test sizes {segment_sizes}, train sizes {[N - s for s in segment_sizes]}."
)

In [ ]:
for target_col in target_cols:
    results_per_split = []
    features = features_stability_reduced_per_split
    for split_idx in range(n_splits):
        train_df, _ = train_splits[split_idx][0]
        test_df, _ = test_splits[split_idx][0]

        # Suppose you already have final features for this target in this split:
        features_for_target = features[split_idx][target_col]

        # Choose alpha (from prior CV, e.g. 0.01)
        enet_alpha = 0.01

        selected_features, results_df = fit_and_compare_models(
            merged_train=train_df,
            merged_test=test_df,
            target_col=target_col,
            feature_list=features_for_target,
            enet_alpha=enet_alpha,
            enet_l1_ratio=0.7,          # optional
            max_feature_fraction=0.15,
        )

        results_per_split.append(results_df)

    results_agg = pd.concat(results_per_split, ignore_index=True)
    summary = results_agg.groupby("model").agg({"pearson_r": "mean", "spearman_r": "mean"}).reset_index()
    print("Mean test metrics across splits:")
    print(target_col)
    print(summary)